# データ構造の理解と pandas 抽出

以下のノートブックでは、添付されている JSON ファイル（P1_4.json, P2_4.json, 4_D.json (P1/P2)）の構造を短く説明し、pandas DataFrame に読み込み、ネストした配列 (例: taskAttempts, agent_output.devices など) をフラット化して抽出します。

- 出力: DataFrame の head() と shape を表示、各抽出テーブルを CSV に保存します。
- ファイルパスはノートブック内で定義しています。Windows 環境向けに絶対パスを使用します。

In [1]:
# 必要ライブラリのインポートとファイル存在チェック
import json
from pathlib import Path
import pandas as pd

# 対象ファイル（workspace の添付ファイルパス）
files = [
    r"c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\ExperimentData\RESULTS\P1\P1_4.json",
    r"c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\ExperimentData\RESULTS\P2\P2_4.json",
    r"c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\InteractiveSmartHome\Assets\EXPERIMENT\VOICE_LOG\P1\4_D.json",
    r"c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\InteractiveSmartHome\Assets\EXPERIMENT\VOICE_LOG\P2\4_D.json",
]

for p in files:
    exists = Path(p).exists()
    print(p, '->', 'FOUND' if exists else 'MISSING')

c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\ExperimentData\RESULTS\P1\P1_4.json -> FOUND
c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\ExperimentData\RESULTS\P2\P2_4.json -> FOUND
c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\InteractiveSmartHome\Assets\EXPERIMENT\VOICE_LOG\P1\4_D.json -> FOUND
c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\InteractiveSmartHome\Assets\EXPERIMENT\VOICE_LOG\P2\4_D.json -> FOUND


In [2]:
# 汎用的な読み込み/フラット化関数

def load_and_flatten(path):
    """
    JSON のトップレベルが list か dict かを判定し、以下の DataFrame を返す dict を返します。
    - 'records': 単純な配列 (list of objects) をフラット化した DataFrame
    - 'tasks': task 単位のメタ情報を持つ DataFrame (存在する場合)
    - 'attempts': 各 task の内部配列 (taskAttempts) を展開した DataFrame (存在する場合)
    """
    path = Path(path)
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    out = {}

    # ケース: JSON がリストで、その各要素が task の集合 (taskAttempts を持つ)
    if isinstance(data, list) and data:
        first = data[0]
        if isinstance(first, dict) and 'taskAttempts' in first:
            # tasks dataframe (top-level fields)
            out['tasks'] = pd.json_normalize(data, sep='.')
            # attempts: taskId を対応付けて展開
            attempts_rows = []
            for task in data:
                tid = task.get('taskId')
                for att in task.get('taskAttempts', []):
                    row = {'taskId': tid}
                    if isinstance(att, dict):
                        for k, v in att.items():
                            row[k] = v
                    else:
                        row['value'] = att
                    attempts_rows.append(row)
            out['attempts'] = pd.json_normalize(attempts_rows, sep='.')
            return out

        # それ以外の list-of-records (例: P1_4.json のような user_prompt 単位)
        out['records'] = pd.json_normalize(data, sep='.')
        return out

    # ケース: JSON が dict (単一オブジェクト)
    if isinstance(data, dict):
        # dict が taskAttempts を持つ場合
        if 'taskAttempts' in data:
            out['tasks'] = pd.json_normalize([{k:v for k,v in data.items() if k!='taskAttempts'}], sep='.')
            attempts = data.get('taskAttempts', [])
            out['attempts'] = pd.json_normalize(attempts, sep='.')
            return out
        # それ以外は top-level を1行の DataFrame として返す
        out['records'] = pd.json_normalize([data], sep='.')
        return out

    # 想定外の型
    return out

In [3]:
# 各ファイルを処理して DataFrame を表示・CSV 保存する
from pathlib import Path
out_base = Path(r"c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\extracted_csvs")
out_base.mkdir(parents=True, exist_ok=True)

results = {}
for p in files:
    ppath = Path(p)
    if not ppath.exists():
        print('SKIP (not found):', p)
        continue
    dfs = load_and_flatten(ppath)
    results[str(ppath)] = {}
    stem = ppath.stem
    for name, df in dfs.items():
        print(f'--- {ppath.name} :: {name} ->', df.shape)
        try:
            display(df.head(5))
        except Exception:
            print(df.head(5))
        # CSV に保存
        out_path = out_base / f"{stem}__{name}.csv"
        try:
            df.to_csv(out_path, index=False, encoding='utf-8-sig')
            print('WROTE', out_path)
        except Exception as e:
            print('FAILED to write', out_path, e)
        results[str(ppath)][name] = {'shape': df.shape, 'csv': str(out_path)}

# 簡易サマリ
import json as _json
print('\nSummary:')
print(_json.dumps(results, ensure_ascii=False, indent=2))

--- P1_4.json :: records -> (22, 14)


,user_prompt,system_total_time_sec,task_id,attempt_id,agent_output.devices,agent_output.response,agent_output.reasoning,metrics.model_name,metrics.tokens.prompt_tokens,metrics.tokens.completion_tokens,metrics.tokens.total_tokens,metrics.cost_usd,metrics.agent_time_elapsed,metrics.system_time_elapsed
0,ボールライト一を緑、ホールライト二を オールライト三を赤につけて,3.751547,2e919b0d-6d84-4641-a535-930a7ca6c8b7,fac90970-eb3b-4a7d-bbbf-df5bb9801945,[{'id': '6e82a90c-d3bb-4e45-b221-ebe21c524b04'...,"Turned on Wall Light 1 in green, Wall Light 2 ...",Matched 'ボールライト一' to 'Wall Light 1' using phon...,gpt-4o-2024-08-06,1831,311,2142,0.007687,3.303510,3.747252
1,ウォールライト二を青に変えて,2.106540,2e919b0d-6d84-4641-a535-930a7ca6c8b7,5dbf45c3-ee64-475f-8cf5-4637e1e3630d,[{'id': '9cbd6e9a-4f7b-41e8-b1dc-464d631dfd7f'...,Changed Wall Light 2 to blue.,Matched 'ウォールライト二' to 'Wall Light 2' using Kat...,gpt-4o-2024-08-06,1818,124,1942,0.005785,1.704266,2.101876
2,で、ウォールライト三を赤に,1.892844,79dc7e7b-d1c7-4237-908d-f541b20406d4,a61c9e99-7fa9-444d-889b-c2fcc71b852e,[{'id': 'badaeb49-9e87-4d7b-882e-be2e80f85c84'...,Turned on Wall Light 3 in red.,Matched 'ウォールライト三' to 'Wall Light 3' using Kat...,gpt-4o-2024-08-06,1818,123,1941,0.005775,1.477890,1.888789
3,セルフライト一、二、四、五、 黄色に付けて,3.982526,fdf65cab-a9a0-4e5f-ac35-c3cff2196644,44402af5-69c7-4615-8dcd-032970541226,[{'id': '409f6dff-d2e2-43c8-8ba5-2492b3e2e5d6'...,"Turned on Shelf Light 1, Shelf Light 2, Shelf ...","Matched 'セルフライト一' to 'Shelf Light 1', 'セルフライト二...",gpt-4o-2024-08-06,1825,394,2219,0.008502,3.590383,3.978898
4,シーリングライト十一をつける,2.154906,2d0b42ad-6703-463c-81b3-4ee4d0c8b347,944b0597-b7ea-4611-b840-dabcb4c46f1c,[{'id': '3da605b5-b12f-4fe8-9108-f4a6648efb3b'...,Turned on Ceiling Light 11.,Matched 'シーリングライト十一' to 'Ceiling Light 11' usi...,gpt-4o-2024-08-06,1818,128,1946,0.005825,1.762425,2.151257


WROTE c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\extracted_csvs\P1_4__records.csv
--- P2_4.json :: records -> (39, 14)


,user_prompt,system_total_time_sec,task_id,attempt_id,agent_output.devices,agent_output.response,agent_output.reasoning,metrics.model_name,metrics.tokens.prompt_tokens,metrics.tokens.completion_tokens,metrics.tokens.total_tokens,metrics.cost_usd,metrics.agent_time_elapsed,metrics.system_time_elapsed
0,ウォールライト一を緑色にしてください,3.383379,2e919b0d-6d84-4641-a535-930a7ca6c8b7,07f19d73-f9ae-4561-9804-a5504ed26db0,[{'id': '6e82a90c-d3bb-4e45-b221-ebe21c524b04'...,Turned on Wall Light 1 in green.,Matched 'ウォールライト一' to 'Wall Light 1' using pho...,gpt-4o-2024-08-06,1819,121,1940,0.005758,2.967165,3.379438
1,ウォールライト二を青色にして下さい。,2.087746,2e919b0d-6d84-4641-a535-930a7ca6c8b7,a7d0bdca-ae11-47a4-83d1-5589aeb1980d,[{'id': '9cbd6e9a-4f7b-41e8-b1dc-464d631dfd7f'...,Turned on Wall Light 2 in blue.,Matched 'ウォールライト二' to 'Wall Light 2' using Kat...,gpt-4o-2024-08-06,1820,126,1946,0.005810,1.669071,2.083994
2,ウォールライト産後、赤くしてくださーい,3.790383,2e919b0d-6d84-4641-a535-930a7ca6c8b7,ced50322-cc1c-4e11-b579-416d3ac77579,[{'id': '6e82a90c-d3bb-4e45-b221-ebe21c524b04'...,"Turned on Wall Light 1, Wall Light 2, and Wall...",Matched 'ウォールライト' to 'Wall Light' using Kataka...,gpt-4o-2024-08-06,1822,275,2097,0.007305,3.391708,3.786603
3,ロールライト一を緑色にしてくださーい,2.047337,2e919b0d-6d84-4641-a535-930a7ca6c8b7,c3495f30-be30-489a-89d0-68611b6371a0,[{'id': '6e82a90c-d3bb-4e45-b221-ebe21c524b04'...,Turned on Wall Light 1 in green.,Matched 'ロールライト一' to 'Wall Light 1' using phon...,gpt-4o-2024-08-06,1823,118,1941,0.005738,1.645035,2.043745
4,ウォールライト二を青くしてください,3.338233,2e919b0d-6d84-4641-a535-930a7ca6c8b7,e32eb413-d00d-49b0-ad11-da742157b909,[{'id': '9cbd6e9a-4f7b-41e8-b1dc-464d631dfd7f'...,Turned on Wall Light 2 in blue.,Matched 'ウォールライト二' to 'Wall Light 2' using Kat...,gpt-4o-2024-08-06,1817,126,1943,0.005803,2.940203,3.334677


WROTE c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\extracted_csvs\P2_4__records.csv
--- 4_D.json :: tasks -> (12, 5)


...

WROTE c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\extracted_csvs\4_D__attempts.csv
--- 4_D.json :: tasks -> (12, 5)


,taskId,taskAttemptCount,totalElapsedTime,finalId,taskAttempts
0,7aeefa77-f398-4a26-b615-e50c4d05ba64,4,83.272860,1f9b5dad-710a-4721-920a-c94951fae31a,[{'attemptId': '00228312-3346-4541-9ef2-e79bea...
1,5dde3eac-8ee6-4821-99d4-f6e4ae5f7bd2,2,7.754818,cdc6954e-1018-4b00-923d-62cc987a8ad6,[{'attemptId': 'cdc6954e-1018-4b00-923d-62cc98...
2,40573e58-0087-4bb3-9ed7-d18a24a35b78,2,6.519046,244ca736-ff65-46ea-943c-4e56f8a6307a,[{'attemptId': '244ca736-ff65-46ea-943c-4e56f8...
3,2e919b0d-6d84-4641-a535-930a7ca6c8b7,2,9.518292,81d3ff94-09fe-4598-aa1a-3db1fc1f3986,[{'attemptId': '81d3ff94-09fe-4598-aa1a-3db1fc...
4,09ff7f59-f6d6-4c54-9955-f1b4bfd11225,2,12.363664,48c0d29b-fe8a-439b-b801-3869b2f3ed42,[{'attemptId': '48c0d29b-fe8a-439b-b801-3869b2...


WROTE c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\extracted_csvs\4_D__tasks.csv
--- 4_D.json :: attempts -> (26, 5)


,taskId,attemptId,taskElapsedTime,userCommand,outputDevices
0,7aeefa77-f398-4a26-b615-e50c4d05ba64,00228312-3346-4541-9ef2-e79bea67d23e,20.48028,エー、テーブルライト、青、フロアライト二を緑,"[6620ef5b-7686-4e7a-9050-2b6c04b8fe77, 4cf87da..."
1,7aeefa77-f398-4a26-b615-e50c4d05ba64,00228312-3346-4541-9ef2-e79bea67d23e,20.48028,エー、テーブルライト、青、フロアライト二を緑,"[6620ef5b-7686-4e7a-9050-2b6c04b8fe77, 4cf87da..."
2,7aeefa77-f398-4a26-b615-e50c4d05ba64,1f9b5dad-710a-4721-920a-c94951fae31a,27.16043,あいうえお,[]
3,7aeefa77-f398-4a26-b615-e50c4d05ba64,1f9b5dad-710a-4721-920a-c94951fae31a,27.16043,あいうえお,[]
4,5dde3eac-8ee6-4821-99d4-f6e4ae5f7bd2,cdc6954e-1018-4b00-923d-62cc987a8ad6,4.239997,こんにちはー,[]


WROTE c:\Users\tenma\Desktop\KeioSchool\InteractiveSmartHome\InteractiveSmartHome\LLMServer\extracted_csvs\4_D__attempts.csv

Summary:
{
  "c:\\Users\\tenma\\Desktop\\KeioSchool\\InteractiveSmartHome\\InteractiveSmartHome\\LLMServer\\ExperimentData\\RESULTS\\P1\\P1_4.json": {
    "records": {
      "shape": [
        22,
        14
      ],
      "csv": "c:\\Users\\tenma\\Desktop\\KeioSchool\\InteractiveSmartHome\\InteractiveSmartHome\\LLMServer\\extracted_csvs\\P1_4__records.csv"
    }
  },
  "c:\\Users\\tenma\\Desktop\\KeioSchool\\InteractiveSmartHome\\InteractiveSmartHome\\LLMServer\\ExperimentData\\RESULTS\\P2\\P2_4.json": {
    "records": {
      "shape": [
        39,
        14
      ],
      "csv": "c:\\Users\\tenma\\Desktop\\KeioSchool\\InteractiveSmartHome\\InteractiveSmartHome\\LLMServer\\extracted_csvs\\P2_4__records.csv"
    }
  },
  "c:\\Users\\tenma\\Desktop\\KeioSchool\\InteractiveSmartHome\\InteractiveSmartHome\\InteractiveSmartHome\\Assets\\EXPERIMENT\\VOICE_LOG\\P1\\

## メモと次の手順

- このノートブックを実行すると、`LLMServer/extracted_csvs` に CSV が出力されます。
- 出力された CSV を確認し、特定の列（例: `agent_output.response`, `agent_output.reasoning`, `metrics.model_name` など）を抽出する追加セルを作成できます。
- 実行して問題が出たら、ここで教えてください。エラーとともに修正します。